In [2]:
import os
import sys
from typing import List, Optional
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5-20251001"  # default model


python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 10


In [46]:
def get_stock_price(symbol: str) -> Optional[float]:
    """
    Get the stock price for a given symbol using the Alpha Vantage API.

    Args:
        symbol (str): The stock symbol to look up.

    Returns:
        Optional[float]: The stock price, or None if not found.
    """
    # Implementation for fetching stock price would go here
    # Return a random float between 100 and 500 for demonstration purposes
    import random
    return round(random.uniform(100, 500), 2)

def get_customer(id): 
    # Implementation for fetching customer information would go here
        return {
            "id": id,
            "name": "John Doe",
            "email": "john.doe@example.com"
        }


def get_account(id): 
        return {
            "id": id,
            "balance": 1000.0,
            "currency": "USD"
        }

In [43]:
# Write a tool schema for the get_stock_price function
get_stock_price_schema = {
    "name": "get_stock_price",
    "description": "Get the stock price for a given symbol. The tool will take stock ticker as input and return the stock price as a float.",
    "input_schema": {
        "type": "object",
        "properties": {
            "symbol": {
                "type": "string",
                "description": "The stock symbol to look up."
            }
        },
        "required": []
    }
}

get_customer_schema = {
    "name": "get_customer",
    "description": "Get information for a given Customer ID. The tool will take exactly 4-digit Customer ID as input and return information as a JSON object.",
    "input_schema": {
        "type": "object",
        "properties": {
            "id": {
                "type": "string",
                "pattern": "^[0-9]{4}$",
                "description": "The Customer ID to look up. This is exactly 4 digits long."
            }
        },
        "required": []
    }
}

get_account_schema = {
    "name": "get_account", 
    "description": "Get information for a given Account ID. The tool will take exactly 8-digit Account ID as input and return information as a JSON object.",
    "input_schema": {
        "type": "object",
        "properties": {
            "id": {
                "type": "string",
                "pattern": "^[0-9]{8}$",
                "description": "The Account ID to look up. This is exactly 8 digits long."
            }
        },
        "required": []
    }
}

In [32]:
from anthropic.types import Message

def add_user_message(messages: List[dict], content) -> None:
    """
    Add a user message to the messages list.

    Args:
        messages (List[dict]): The list of messages.
        content (str): The content of the user message.
    """
    if isinstance(content, Message):
        messages.append({"role": "user", "content": content.content})
    else:
        messages.append({"role": "user", "content": content})

def add_assistant_message(messages: List[dict], content) -> None:
    """
    Add an assistant message to the messages list.

    Args:
        messages (List[dict]): The list of messages.
        content (str): The content of the assistant message.
    """
    if isinstance(content, Message):
        messages.append({"role": "assistant", "content": content.content})
    else:
        messages.append({"role": "assistant", "content": content})

def converse(
    messages: List[dict],
    model: str = model,
    max_tokens_to_sample: int = 300,
    temperature: Optional[float] = None,
    stop_sequences: Optional[List[str]] = None,
    tools: Optional[List[dict]] = None,
    system_prompt: Optional[str] = None
) -> str:
    """
    Converse with the Claude model.

    Args:
        messages (List[dict]): A list of message dictionaries.
        model (str): The model to use for the conversation.
        max_tokens_to_sample (int): The maximum number of tokens to sample.
        temperature (Optional[float]): The sampling temperature.
        stop_sequences (Optional[List[str]]): A list of stop sequences.
        tools (Optional[List[dict]]): A list of tools available for the model.
        system_prompt (Optional[str]): A system prompt to include in the conversation.

    Returns:
        str: The response from the Claude model.
    """
    arg_list = {
        # get model from environment variable if set, otherwise use the default model
        "model": model,
        "messages": messages,
        "max_tokens": 1024,
        "thinking": {"type": "disabled"},
    }

    if temperature is not None:
        arg_list["temperature"] = temperature

    if stop_sequences is not None:
        arg_list["stop_sequences"] = stop_sequences
    if tools is not None:
        arg_list["tools"] = tools
    if system_prompt is not None:
        arg_list["system"] = system_prompt
    
    # print(arg_list)

    response = client.messages.create(**arg_list)
        
    return response

In [26]:
import os
key = os.environ.get("ANTHROPIC_API_KEY", "")
print(len(key), key[:12])

108 sk-ant-api03


In [44]:
#Run it 20 times for temperature setting as 1.0
# Open a File to append all the responses: 
ids = [{"prompt": "account", "id": 1234}, {"prompt": "customer", "id": 12345678}]
temp = [0.0, 0.5, 1.0]
for id_dict in ids:
    prompt = id_dict["prompt"]
    id = id_dict["id"]
    for temp_val in temp:
        with open(f"responses_regex_{temp_val}_{id}.txt", "a") as f:
            for i in range(20):
                messages = []
                add_user_message(messages, f"Lookup {prompt} ID {id}.")
                response = converse(messages, tools=[get_stock_price_schema, get_customer_schema, get_account_schema], temperature=temp_val)
                resp_temp = response
                f.write(f"Response {i+1}: {resp_temp}\n") 


In [58]:
def response_parser(response):
    """
    Parse the response from the Claude model.

    Args:
        response: The response object from the Claude model.

    Returns:
        The parsed response.
    """
    claude_response = {}
    if response is None:
        return None
    if isinstance(response, Message):
        claude_response["message_id"] = response.id
        claude_response["stop_reason"] = response.stop_reason
        if response.container is not None:
            claude_response["container"] = response.container
        if response.content is not None:
            contents = response.content
            claude_response["contents"] = []
            for content in contents:
                if content.type == "text":
                    claude_response["contents"].append({"completion": content.text})
                elif content.type == "tool_use":
                    claude_response["contents"].append({"tool_id": content.id, "input": content.input, "tool_name": content.name})
                    
    return claude_response

response_parsed = response_parser(response)
response_parsed

{'message_id': 'msg_011CeJnhyFgCoQXotKPPBs5K',
 'stop_reason': 'end_turn',
 'contents': [{'completion': 'I need to look up an account, but the Account ID must be exactly 8 digits long. The ID you provided (1234) is only 4 digits. \n\nCould you please provide the complete 8-digit Account ID?'}]}

In [68]:

messages = [{'role': 'user', 'content': "Lookup account with ID 1234"}]
response = converse(messages, tools=[get_stock_price_schema, get_customer_schema, get_account_schema], temperature=0.0)
add_assistant_message(messages, response.content)
# messages
messages_temp = messages.copy()
messages_temp

[{'role': 'user', 'content': 'Lookup account with ID 1234'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll look up the account with ID 1234 for you.", type='text'),
   ToolUseBlock(id='toolu_01QM2un9a5Hqj822ixEVfNMJ', caller=DirectCaller(type='direct'), input={'id': '1234'}, name='get_account', type='tool_use')]}]

In [70]:
import re

SCHEMA_PATTERN = {
    "get_customer": r"^\d{4}$",
    "get_account": r"^\d{8}$"
}

def validate_tool_input(tool_name: str, tool_input: dict) -> bool:
    """
    Validate the tool input based on the tool name and input schema.

    Args:
        tool_name (str): The name of the tool to validate.
        tool_input (dict): The input parameters for the tool.

    Returns:
        bool: True if the input is valid, False otherwise.
    """
    if tool_name in SCHEMA_PATTERN:
        pattern = SCHEMA_PATTERN[tool_name]
        for key, value in tool_input.items():
            if not re.match(pattern, value):
                return False
    return True

def execute_tools(tool_name: str, tool_input: dict) -> str:
    """
    Execute a tool based on the tool name and input.

    Args:
        tool_name (str): The name of the tool to execute.
        tool_input (dict): The input parameters for the tool.

    Returns:
        dict: The result of the tool execution.
    """
    if tool_name == "get_stock_price":
        symbol = tool_input.get("symbol")
        if symbol:
            price = get_stock_price(symbol)
            return f"${price}" 
        else:
            return "Symbol not provided"
    elif tool_name == "get_customer":
        customer_id = tool_input.get("id")
        if customer_id and validate_tool_input("get_customer", tool_input):
            customer_info = get_customer(customer_id)
            return customer_info
        else:
            return "Customer ID not provided"
    elif tool_name == "get_account":
        account_id = tool_input.get("id")
        if account_id and validate_tool_input("get_account", tool_input):
            account_info = get_account(account_id)
            return account_info
        else:
            return "Account ID not provided or not in valid format"
    else:
        return f"Tool '{tool_name}' not recognized"

user_message_content = []
response_parsed = response_parser(response)
# Check if there is tool use in the response before executing
if response_parsed and "contents" in response_parsed and len(response_parsed["contents"]) > 0 and "tool_id" in response_parsed["contents"][0]:
    user_message_content.append({"type": "tool_result", "tool_use_id": response_parsed["contents"][0]["tool_id"], 
    "content": execute_tools(response_parsed["contents"][0]["tool_name"], response_parsed["contents"][0]["input"])})
    add_user_message(messages_temp, user_message_content)
    print(messages_temp)
#Check if textblock returned instead of tool use
elif response_parsed and "contents" in response_parsed and len(response_parsed["contents"]) > 0 and "completion" in response_parsed["contents"][0]:
    user_message_content.append({"type": "textblock", "content": response_parsed["contents"][0]["completion"]})
    
    print(messages_temp)
else:
    print(f"No tool use or textblock found in the response.{response_parsed}")

[{'role': 'user', 'content': 'Lookup account with ID 1234'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll look up the account with ID 1234 for you.", type='text'), ToolUseBlock(id='toolu_01QM2un9a5Hqj822ixEVfNMJ', caller=DirectCaller(type='direct'), input={'id': '1234'}, name='get_account', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'textblock', 'content': "I'll look up the account with ID 1234 for you."}]}]


In [65]:
response2 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    thinking={"type": "disabled"},
    tools=[get_stock_price_schema],
    messages=messages_temp
)

response2 

Message(id='msg_011Ce5ER55fQFFRViLaEdeER', container=None, content=[TextBlock(citations=None, text="Apple's stock price (AAPL) is currently **$295.97**.", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=676, output_tokens=21, output_tokens_details=None, server_tool_use=None, service_tier='standard'))